# Lab 2: GPT from scratch

In this lab, you will dive into the inner workings of the GPT architecture. You will walk through a complete implementation of the architecture in PyTorch, instantiate this implementation with pre-trained weights, and put the resulting model to the test by generating text. At the end of this lab, you will understand the building blocks of the GPT architecture and how they are connected.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

In [1]:
from dataclasses import dataclass

import torch
import torch.nn as nn

## Part 1: GPT architecture

GPT-2 was first described by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). To faithfully implement the model, one needs to also read the earlier paper by [Radford et al. (2018)](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf). Another important source of information is the code released by OpenAI, which is available on GitHub ([link](https://github.com/openai/gpt-2)).

The GPT architecture is made up of a stack of Transformer blocks. Each block has two main parts: one handles multi-head self-attention, and the other is a feed-forward network. Before these parts do their work, their input undergoes layer normalisation, and residual connections are added to help the model learn more effectively. The input to the architecture is a sequence of token IDs; these are turned into embeddings and augmented with information about the absolute position of each token in the sequence. The output layer converts the internal representations into logit scores for every token in the vocabulary.

### Model configuration

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) present four increasingly larger GPT models based on the same architecture. Here, we will implement the smallest of these, characterised by the following hyperparameters:

In [2]:
@dataclass
class Config:
    n_vocab: int = 50_257
    n_ctx: int = 1024
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12

#### 🧩 Task 2.01: Model configuration

Explain the purpose of these hyperparameters. In particular, where does the number 50,257 come from?

#### ANSWER

`n_vocab = 50,257` is the size of GPT-2's vocabulary. It consists of 256 byte-level tokens, 50,000 tokens introduced through BPE merges, and one special `<|endoftext|>` token.

`n_ctx = 1024` is the maximum number of tokens the model can process in one context.

`n_embd = 768` is the dimensionality of each token representation.

`n_head = 12` is the number of attention heads. Since the embedding dimension is 768, each head has dimension \(768 / 12 = 64\).

`n_layer = 12` is the number of Transformer decoder blocks stacked in the model.

### GELU activation function

We start by implementing the feed-forward network. This is a standard two-layer network with a Gaussian Error Linear Unit (GELU) activation function ([Hendrycks and Gimpel, 2016](https://doi.org/10.48550/arXiv.1606.08415)).

The GELU is a smooth version of the rectified linear unit (ReLU) that weights inputs by their value under the cumulative distribution function of the standard Gaussian. This function is commonly denoted by $\Phi$. For example, $\text{GELU}(0{.}5) = 0{.}5 \cdot \Phi(0{.}5) \approx 0{.}5 \cdot 0{.}6915 = 0{.}3457$ because approximately 69.15% of normally distributed data lies to the left of $0{.}5$.

When GPT-2 was released, computing the GELU exactly was expensive, and the released code therefore used an approximation originally presented by [Page (1977)](https://doi.org/10.2307/2346872). We follow suit here, as we want to create a replica of the original model. However, it is worth mentioning that PyTorch now offers an exact implementation of the GELU so fast that using an approximation is unnecessary.

In [3]:
def gelu(x):
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x**3)))

#### 🎓 Task 2.02: Mathematical properties of the GELU

Find the minimal output value of the (approximated) GELU and the input value for which it yields that output. Use a service such as [WolframAlpha](https://www.wolframalpha.com/) for the necessary derivations. What are the main differences between the GELU and the ReLU?


#### ANSWER

The approximated GELU used in GPT-2 is

$$
\mathrm{GELU}(x)
=
\frac{1}{2}x
\left(
1+\tanh\left(
\sqrt{\frac{2}{\pi}}
\left(x+0.044715x^3\right)
\right)
\right).
$$

Its minimum occurs approximately at

$$
x \approx -0.7525,
$$

where

$$
\mathrm{GELU}(x) \approx -0.1700.
$$

The main difference between GELU and ReLU is how they treat negative inputs.

ReLU is defined as

$$
\mathrm{ReLU}(x)=\max(0,x).
$$

Therefore, every negative input is mapped exactly to zero. GELU instead smoothly scales the input according to its magnitude, so small negative inputs can still produce small negative outputs.

Main differences:

- ReLU has a sharp transition at $x=0$, while GELU is smooth.
- ReLU completely removes negative values, while GELU can preserve some negative information.
- For large positive inputs, both functions behave approximately like $x$.
- For large negative inputs, both approach zero, but GELU approaches zero smoothly from below.
- GELU is differentiable everywhere, while ReLU is not differentiable at $x=0$.

### Feed-forward network

Next, here is the code for the feed-forward network. Note that we follow the released code and use the name **multi-layer perceptron (MLP)** rather than “feed-forward network”.

In [6]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)

    def forward(self, x):
        # x shape: [batch_size, seq_len, n_embd]

        batch_size, seq_len, n_embd = x.shape

        x = self.c_fc(x)
        # x shape: [batch_size, seq_len, 4 * n_embd]
        # The linear layer transforms only the last dimension:
        # n_embd -> 4 * n_embd

        x = gelu(x)
        # x shape: [batch_size, seq_len, 4 * n_embd]
        # GELU is applied element-wise, so the shape does not change.

        x = self.c_proj(x)
        # x shape: [batch_size, seq_len, n_embd]
        # The second linear layer projects:
        # 4 * n_embd -> n_embd

        return x
        # output shape: [batch_size, seq_len, n_embd]

#### 🎓 Task 2.03: Shape annotations

One of the most common errors in deep learning is a mismatch in tensor dimensions. To avoid this, it is good practice to annotate PyTorch code with shapes. For example, suppose you are given the following code:

In [8]:
f = nn.Linear(5, 7)
x = torch.rand(2, 3, 5)
y = f(x)

The annotation of this code with shapes would look as follows:

In [9]:
f = nn.Linear(5, 7)
# not a tensor variable; needs no annotation

x = torch.rand(2, 3, 5)
# shape of x: [2, 3, 5]

y = f(x)
# shape of y: [2, 3, 7]

Annotate the shapes in the `forward()` method of the feed-forward network. Instead of using actual numbers, refer to dimension sizes by symbolic names such as `n_embd`, `batch_size` (number of samples in a batch of input data) and `seq_len` (length of an input sequence). You can introduce additional names and other notation you find useful. Make your annotations as detailed as you need them to explain how the shapes change from one line to the next.

### Causal mask

Our next goal is to implement the core of the GPT architecture: the multi-head attention mechanism.

Recall that the attention mechanism in the Transformer decoder must be restricted to attending only to previously generated tokens. This type of attention is also called **causal attention**. In practice, we implement it through a masking technique that sets the post-softmax attention weights of future tokens to zero. The following utility function implements such a mask:

In [10]:
def make_causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

#### 🧩 Task 2.04: Causal mask

Have a close look at the following code and run it to see the result. What are the shapes of `x` and `mask`? Given that the shapes are different, why does the addition operation in the last line not raise an error? What is the shape of the result?

How does the addition operation implement masking? (Recall that the attention scores are normalised using the softmax function.)


#### ANSWER

`x` has shape `[1, 2, 3, 3]`, while `mask` has shape `[5, 5]`. After slicing, `mask[:3, :3]` has shape `[3, 3]`.

The addition works because PyTorch uses **broadcasting**: the mask is applied across the first two dimensions of `x`. The result has shape `[1, 2, 3, 3]`.

The mask contains `0` on and below the diagonal and $-\infty$ above it. Adding $-\infty$ to future-token attention scores makes their softmax probability equal to 0, preventing attention to future positions.


In [11]:
x = torch.rand(1, 2, 3, 3)
mask = make_causal_mask(5)
x + mask[:3, :3]

tensor([[[[0.1499,   -inf,   -inf],
          [0.6541, 0.4209,   -inf],
          [0.3671, 0.6798, 0.4166]],

         [[0.3844,   -inf,   -inf],
          [0.4188, 0.8241,   -inf],
          [0.2695, 0.2025, 0.2869]]]])

### Attention mechanism

Here is the code for the multi-head attention mechanism:

In [ ]:
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.register_buffer("mask", make_causal_mask(config.n_ctx), persistent=False)

    def forward(self, x):
        # x: [batch_size, seq_len, n_embd]

        batch_size, seq_len, n_embd = x.shape
        head_embd = n_embd // self.n_head

        q, k, v = self.c_attn(x).chunk(3, dim=-1)
        # q, k, v: [batch_size, seq_len, n_embd]

        q = q.view(batch_size, seq_len, self.n_head, head_embd)
        k = k.view(batch_size, seq_len, self.n_head, head_embd)
        v = v.view(batch_size, seq_len, self.n_head, head_embd)
        # q, k, v: [batch_size, seq_len, n_head, head_embd]

        q = q.transpose(-2, -3)
        k = k.transpose(-2, -3)
        v = v.transpose(-2, -3)
        # q, k, v: [batch_size, n_head, seq_len, head_embd]

        x = q @ k.transpose(-1, -2)
        # x: [batch_size, n_head, seq_len, seq_len]

        x = x / head_embd**0.5
        # same shape

        x = x + self.mask[:seq_len, :seq_len]
        # x: [batch_size, n_head, seq_len, seq_len]
        # broadcasting happens here

        x = torch.softmax(x, dim=-1)
        # same shape

        x = x @ v
        # x: [batch_size, n_head, seq_len, head_embd]

        x = x.transpose(-2, -3).contiguous()
        # x: [batch_size, seq_len, n_head, head_embd]

        x = x.view(batch_size, seq_len, n_embd)
        # x: [batch_size, seq_len, n_embd]

        x = self.c_proj(x)
        # x: [batch_size, seq_len, n_embd]

        return x

#### 🎓 Task 2.05: Multi-head attention

Trace the input `x` through the `forward()` method line by line and annotate the shapes of all tensor variables. Identify all lines that rely on broadcasting.

### Layer normalisation

As mentioned above, the inputs to both the feed-forward network and the multi-head attention mechanism undergo **layer normalisation**. This normalises the inputs to have zero mean and unit variance across the activations. [Ba et al. (2016)](https://doi.org/10.48550/arXiv.1607.06450) introduce two trainable parameters (called $\gamma$ and $\beta$ in the paper) that allow the network to learn an appropriate scale and shift for the normalised values.

We implement layer normalisation as follows:

In [13]:
class LayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.g = nn.Parameter(torch.ones(config.n_embd))
        self.b = nn.Parameter(torch.zeros(config.n_embd))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(unbiased=False, dim=-1, keepdim=True)
        return self.g * (x - mean) / torch.sqrt(variance + 1e-05) + self.b

#### 🧩 Task 2.06: Layer normalisation

What is the relevance of the `keepdim=True` keyword argument in the `mean()` and `var()` functions? What would happen if we omitted it?

What is the relevance of the constant 1e-05? What could happen if we omitted it?

#### ANSWER


`keepdim=True` keeps the reduced dimension with size 1.

For an input with shape `[batch_size, seq_len, n_embd]`, the mean and variance therefore have shape:

`[batch_size, seq_len, 1]`

This allows PyTorch to broadcast them correctly across the embedding dimension when normalising `x`.

If `keepdim=True` were omitted, the shape would become `[batch_size, seq_len]`, which would not broadcast correctly with `x`.

The constant `1e-05` is added for numerical stability. If the variance were zero or extremely small, dividing by `sqrt(variance)` could cause division by zero or very large numerical values. Adding `1e-05` prevents this.

### Decoder block

We now combine the feed-forward network, the multi-head attention mechanism and the layer normalisation into a decoder block.

In [14]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config)
        self.attn = Attention(config)
        self.ln_2 = LayerNorm(config)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

#### 🎓 Task 2.07: Pre-norm and post-norm architectures

The original Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) is a “post-norm architecture”, where the normalisation is applied **after** each residual block. In contrast, GPT-2 is a “pre-norm architecture”, where the normalisation is applied **before**. Find the passage in Section&nbsp;2.3 of [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) that reports on this modification.

[Xiong et al. (2020)](https://arxiv.org/pdf/2002.04745) compare pre-norm and post-norm architectures empirically. Read the abstract of their paper and summarise their main findings. According to these findings, what are the benefits of the pre-norm architecture?


#### ANSWER

In GPT-2, layer normalization was moved to the **input of each sub-block**. This means normalization is applied before the attention or MLP operation, making GPT-2 a **pre-norm** Transformer.

The original Transformer uses **post-norm**, where normalization is applied after the residual connection.

According to Xiong et al. (2020), post-norm Transformers can have very large gradients near the output layers at initialization, which can make training unstable and often requires learning-rate warm-up.

Pre-norm Transformers have more stable gradients at initialization. Therefore, they are generally easier to train and require less learning-rate warm-up and hyperparameter tuning.

### Model

We now have almost all components in place to complete the implementation of the GPT-2 model. The only thing  missing are the position embeddings. These simply associate an embedding vector with every position in the context window. To set them up, we first define another utility function:

In [15]:
def make_positions(n):
    return torch.arange(n, dtype=torch.long)

We then code the complete model as follows:

In [16]:
class Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.n_vocab, config.n_embd)
        self.wpe = nn.Embedding(config.n_ctx, config.n_embd)
        self.h = nn.Sequential(*(Block(config) for _ in range(config.n_layer)))
        self.ln_f = LayerNorm(config)
        self.lm_head = nn.Linear(config.n_embd, config.n_vocab, bias=False)
        self.register_buffer("pos", make_positions(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len = x.shape
        wte = self.wte(x)
        wpe = self.wpe(self.pos[:seq_len])
        x = wte + wpe
        x = self.h(x)
        x = self.ln_f(x)
        x = self.lm_head(x)
        return x

#### 🧩 Task 2.08: Buffers

Our implementation registers the vector of positions as a buffer. (Earlier, we also registered the causal mask as a buffer.) Consult the PyTorch documentation to determine the benefits of registering a tensor as a buffer, in contrast to computing it in the `forward()` method.


#### ANSWER

Registering a tensor as a **buffer** means that it becomes part of the model state, but it is **not a trainable parameter**.

This is useful for tensors such as the causal mask and position indices because:

- they are automatically moved with the model when using `.to(device)`, `.cuda()`, etc.
- they can be saved and loaded together with the model state
- they are not updated by the optimizer

If these tensors were created inside `forward()`, they would have to be recreated every time the model is called and would need to be placed on the correct device manually.

#### 🎓 Task 2.09: Number of trainable parameters

The model we have implemented is the smallest one presented by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). But how many trainable parameters exactly does it have? Interestingly, the number originally reported by the authors is wrong! What number did they report?

Your task is to write code to compute the number of parameters yourself. This should only take 1–3 lines of code. What number do you get when you apply this code to a fresh model instance?

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) followed the original Transformers paper ([Vaswani et al., 2017](https://doi.org/10.48550/arXiv.1706.03762)) and shared the trainable weights between the token embedding and the final linear layer. Implement this weight sharing strategy. (Hint: This only requires one line of code.) Then, re-compute the number of trainable parameters for the modified model. What number do you get now? How large is the reduction caused by the weight sharing?

In [17]:
model = Model(Config())

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {n_params:,}")

Number of trainable parameters: 163,037,184


In [18]:
model.lm_head.weight = model.wte.weight

n_params_shared = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters with weight sharing: {n_params_shared:,}")

Number of trainable parameters with weight sharing: 124,439,808


#### ANSWER - Task 2.09

The GPT-2 paper reports **117M parameters** for the smallest model.

Our model initially has **163,037,184 trainable parameters** because the token embedding and output layer have separate weights.

After sharing the weights with

`model.lm_head.weight = model.wte.weight`

the model has **124,439,808 trainable parameters**.

The reduction is

$$
163,037,184 - 124,439,808 = 38,597,376
$$

parameters, which is about **23.7%** of the original untied model.

## Part 2: Load pre-trained weights

Now that you have a complete implementation of the GPT-2 model in place, you can instantiate it by loading the pre-trained weights released by OpenAI. These weights were originally provided in the TensorFlow format. For this lab, we have re-packaged them as a single file in NumPy’s `.npz` archive format. We can load it as follows:

In [21]:
import numpy as np

pretrained = np.load("gpt-2-pretrained.npz")

The result `pretrained` is a dictionary mapping names to NumPy arrays. When you print the names, you will see that they correspond to the attributes of our network modules, even though the names differ. For example, the array `h0.attn.c_attn.b` holds the biases (`b`) of the `c_attn` linear layer in the attention mechanism (`attn`) of the first transformer block (`h0`).

#### 🎓 Task 2.10: Load pre-trained weights

Create a model from the pre-trained weights. To do this, you need to instantiate a fresh model and write the contents of each array from the `npz` archive with the pre-trained weights into the corresponding tensor. To make this a bit easier, here is a utility function that copies data from a NumPy array `source` to a PyTorch tensor `target`:

In [22]:
def copy_weights(source: np.ndarray, target: torch.Tensor):
    assert source.shape == target.shape
    with torch.no_grad():
        target.copy_(torch.tensor(source, dtype=torch.float32))

You can start from this skeleton code:

In [32]:
def from_pretrained() -> Model:
    model = Model(Config())
    pretrained = np.load("gpt-2-pretrained.npz")

    # Token and positional embeddings
    copy_weights(pretrained["wte"], model.wte.weight)
    copy_weights(pretrained["wpe"], model.wpe.weight)

    # Transformer blocks
    for i, block in enumerate(model.h):
        prefix = f"h{i}"

        # Layer norms
        copy_weights(pretrained[f"{prefix}.ln_1.g"], block.ln_1.g)
        copy_weights(pretrained[f"{prefix}.ln_1.b"], block.ln_1.b)
        copy_weights(pretrained[f"{prefix}.ln_2.g"], block.ln_2.g)
        copy_weights(pretrained[f"{prefix}.ln_2.b"], block.ln_2.b)

        # Attention
        # Linear-layer weights must be transposed for PyTorch
        copy_weights(pretrained[f"{prefix}.attn.c_attn.w"].T, block.attn.c_attn.weight)
        copy_weights(pretrained[f"{prefix}.attn.c_attn.b"], block.attn.c_attn.bias)

        copy_weights(pretrained[f"{prefix}.attn.c_proj.w"].T, block.attn.c_proj.weight)
        copy_weights(pretrained[f"{prefix}.attn.c_proj.b"], block.attn.c_proj.bias)

        # MLP
        copy_weights(pretrained[f"{prefix}.mlp.c_fc.w"].T, block.mlp.c_fc.weight)
        copy_weights(pretrained[f"{prefix}.mlp.c_fc.b"], block.mlp.c_fc.bias)

        copy_weights(pretrained[f"{prefix}.mlp.c_proj.w"].T, block.mlp.c_proj.weight)
        copy_weights(pretrained[f"{prefix}.mlp.c_proj.b"], block.mlp.c_proj.bias)

    # Final layer norm
    copy_weights(pretrained["ln_f.g"], model.ln_f.g)
    copy_weights(pretrained["ln_f.b"], model.ln_f.b)

    # GPT-2 shares token embedding and output-layer weights
    model.lm_head.weight = model.wte.weight

    return model

model = from_pretrained()
print("Pretrained model loaded successfully!")

Pretrained model loaded successfully!


**Important:** One technical detail to note is that PyTorch stores the weights of linear layers in a transposed form. For example, a linear layer created as `nn.Linear(2, 3)` has a weight matrix of shape [3, 2].

## Part 3: Put the model to use

In the third and final part of this lab, you will use the pre-trained model to generate text and evaluate it on a standard benchmark.

### Sampling-based text generation

The easiest way to generate text with a language model is by using a **greedy approach**. This method works by creating text one token at a time. At each step, the model takes the previously generated text (called the **context**) as input and adds the token with the highest output logit as a new token. The code in the next cell defines a function `generate()` that forms the core of a greedy generator:

In [34]:
def generate(model, context, context_size=1024, n_tokens=20):
    for _ in range(n_tokens):
        context = context[:, -context_size:]
        with torch.no_grad():
            logits = model(context)[:, -1, :]
        next_idx = torch.argmax(logits, dim=-1, keepdim=True)
        context = torch.cat([context, next_idx], dim=-1)
    return context

To use this function with an actual text input, you need a tokeniser to first encode the text into a vector of token IDs, and later decode the generated `context` into new text. The reference implementation of the GPT-2 tokeniser is in the library `tiktoken`. The code in the next cell sets up the tokeniser, loads the pretrained model from Task&nbsp;2.10, and then defines a helper function that handles the encoding and decoding.

In [35]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
model = from_pretrained()


def generate_helper(text, context_size=1024, n_tokens=20):
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)
    context = generate(model, context, context_size=context_size, n_tokens=n_tokens)
    return tokenizer.decode(context[0].tolist())

You can use this helper function to generate text as follows:

In [36]:
generate_helper("Linköping University is")

"Linköping University is a research institute in the Netherlands.\n\nThe research was funded by the European Union's European Research"

**Tip:** If you did not manage to complete Task&nbsp;2.10, you can still work on this task by using a pretrained GPT-2 model from [Hugging Face](https://huggingface.co/openai-community/gpt2). The next code cell shows how you would instantiate this model. Note that you may have to first install the `transformers` library.

In [ ]:
# from transformers import GPT2LMHeadModel
# model = GPT2LMHeadModel.from_pretrained("gpt2")
# logits = model(context).logits[:, -1, :]

#### 🎓 Task 2.11: Sampling-based text generation

 The greedy approach to text generation is not very interesting for practical applications because it always chooses the most likely token, leading to predictable and less creative results. Your task is to modify the code for the `generate()` function to use a **sampling-based approach** instead. In this approach, the next token is chosen randomly based on the probabilities assigned by the model (softmax-normalised logits), treating them as a categorical distribution over the token vocabulary. Additionally, your code should include two common techniques to improve sampling:
 
 * **temperature scaling**, which lets the user control the randomness of the sampling
 * **top-$k$ sampling**, which limits the sampling to the top-$k$ most likely tokens, ignoring less probable ones

In [40]:
def generate(model, context, context_size=1024, n_tokens=20, temperature=1.0, top_k=None):
    for _ in range(n_tokens):
        context = context[:, -context_size:]

        with torch.no_grad():
            logits = model(context)[:, -1, :]

        # Temperature scaling
        logits = logits / temperature

        # Keep only the top-k most likely tokens
        if top_k is not None:
            top_values, _ = torch.topk(logits, top_k)
            threshold = top_values[:, -1].unsqueeze(-1)
            logits = torch.where(
                logits < threshold,
                torch.full_like(logits, float("-inf")),
                logits,
            )

        # Convert logits to probabilities
        probs = torch.softmax(logits, dim=-1)

        # Sample the next token from the probability distribution
        next_idx = torch.multinomial(probs, num_samples=1)

        context = torch.cat([context, next_idx], dim=-1)

    return context


def generate_helper(
    text,
    context_size=1024,
    n_tokens=20,
    temperature=1.0,
    top_k=None,
):
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)

    context = generate(
        model,
        context,
        context_size=context_size,
        n_tokens=n_tokens,
        temperature=temperature,
        top_k=top_k,
    )

    return tokenizer.decode(context[0].tolist())

generate_helper(
    "Linköping University is",
    n_tokens=50,
    temperature=0.8,
    top_k=5,
)

"Linköping University is a leading provider of high-speed Internet services for all ages. The University is also home to the world's largest library of online courses.<|endoftext|>This is a guest post from my colleague and friend, David K. Smith.\n\nIn this post"

#### REMARK - Task 2.11


Instead of always choosing the token with the highest logit, the model converts the logits into probabilities using softmax and samples the next token from this probability distribution.

**Temperature scaling** controls how random the sampling is:

- temperature $< 1$ makes the distribution sharper and the output more predictable
- temperature $> 1$ makes the distribution flatter and the output more random

**Top-$k$ sampling** keeps only the $k$ most likely tokens and sets all other logits to $-\infty$, so their probability becomes 0.

Finally, `torch.multinomial()` samples the next token according to the resulting probabilities.

### Evaluating the pretrained model

If you have experimented with your pretrained GPT-2 model, you will have noticed that its ability to generate useful text is somewhat limited. By today’s standards, GPT-2 is a small model with modest capabilities. However, it can still be helpful for certain tasks, such as text autocompletion, generating filler text, or answering simple questions. To rigourosly evaluate language models, researchers often use standard benchmark datasets. Creating these benchmarks is a discipline of its own, and they tend to become increasingly challenging as models continue to improve.

In the final task of this lab, you will evaluate GPT-2’s performance on a small subset of the [HellaSwag dataset](https://rowanzellers.com/hellaswag/), which was published in the same year as GPT-2 itself (2019). HellaSwag is designed to test a model’s ability to perform commonsense reasoning in challenging contexts. Unlike simpler benchmarks, HellaSwag presents scenarios where the correct text completion depends on semantic relationships between events and on world knowledge. This makes it a good choice for assessing the ability of language models to go beyond surface-level patterns and produce meaningful, context-aware predictions.

#### 🎓 Task 2.12: Evaluating the pretrained model

Read the [HellaSwag website](https://rowanzellers.com/hellaswag/) to get some background on the benchmark. How does a sample from the dataset look like? What is an expected prediction? How does the benchmark allow us to score models? What is the random baseline? What is the human performance reported on the task?

The next cell contains code for evaluating your pretrained model on a small sample from HellaSwag. You will also need a tokenizer. The HellaSwag subset is in the file `hellaswag-mini.jsonl`. Inspect that file to understand the format. Next, read the code and explain how it works. Specifically, how does the code compute the score of individual endings? In the call to `cross_entropy()`, why are the tensors sliced in this specific way?

Finally, what overall score does the pretrained GPT-2 model get on this benchmark? How does that score compare to the random baseline and the human performance?

In [43]:
## Inspect one HellaSwag sample

import json

with open("hellaswag-mini.jsonl") as f:
    sample = json.loads(next(f))

print("Context:")
print(sample["ctx"])

print("\nPossible endings:")
for i, ending in enumerate(sample["endings"]):
    print(f"{i}: {ending}")

print("\nCorrect label:", sample["label"])

Context:
[header] How to paint jeans [title] Cover your work area. [step] Protect your surfaces from the paint or it may stain them. Make sure to cover a large area.

Possible endings:
0: You will be splattering the paint onto your jeans and it is likely to splatter all around your jeans as well. Lay out your tarp or newspapers to cover the area you will be working on.
1: You might want to put out a tarp or blanket for added protection. Do not make the area wet; any paint sitting around your work area could stain your floor.
2: A single spill can permanently stain denim if it gets onto your skin, so make sure to have everything else covered. [substeps] Get accumulation of lots of cleaning substances underneath your work surface.
3: Cover the top areas of your work area with paper towels or another material. Use painter's tape or duct tape.

Correct label: 0


In [41]:
import json

with open("hellaswag-mini.jsonl") as f:
    n_correct = 0
    n_total = 0
    for line in f:
        sample = json.loads(line)
        prefix = tokenizer.encode(sample["ctx"])
        ending_scores = []
        for i, ending in enumerate(sample["endings"]):
            suffix = tokenizer.encode(" " + ending)
            context = torch.tensor([prefix + suffix], dtype=torch.long)
            with torch.no_grad():
                logits = model(context)
                ending_score = torch.nn.functional.cross_entropy(
                    logits[0, -len(suffix) - 1 : -1], context[0, -len(suffix) :]
                )
            ending_scores.append((ending_score, i))
        predicted = min(ending_scores)[1]
        n_correct += int(predicted == sample["label"])
        n_total += 1
    print(f"Accuracy: {n_correct / n_total:.2%}")

Accuracy: 30.47%


####  Answer - Task 2.12

HellaSwag is a multiple-choice commonsense reasoning benchmark. Each sample contains a context followed by four possible endings, and the model must choose the most plausible continuation.

Since there are four possible endings, the random baseline is **25%**. Reported human performance on HellaSwag is **95.6%**.

For each possible ending, the code joins the context and the ending and passes the complete sequence through GPT-2. It then computes the cross-entropy loss only for the tokens belonging to the ending.

The slices

`logits[0, -len(suffix) - 1 : -1]`

and

`context[0, -len(suffix) :]`

are shifted by one position because GPT-2 predicts the **next token**. Each logit at position $t$ is therefore compared with the token at position $t+1$.

The ending with the **lowest cross-entropy loss** is selected because a lower loss means that GPT-2 assigns a higher probability to that ending.

The pretrained GPT-2 model achieves **30.47% accuracy** on the provided HellaSwag subset. This is slightly above the **25% random baseline**, but far below the **95.6% human performance**.

**🥳 Congratulations on finishing lab&nbsp;2!**